# 04 - Build the analysis panel

This notebook builds the panel organized by researcher, conference, and year from the
cleaned Step 1 PC service data, the Step 2 citation graph, and the Step
3 identity matches.

The main public output is a panel with one row per researcher,
conference, and year. Feature table stores the matching,
service history, first service, and alternative outcome fields. The
panel keeps all PC researchers, but citation outcomes are not filled
for the small set of unresolved manual review cases.

## 1. Setup

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

import os
import sys
from pathlib import Path

os.environ.setdefault("ARROW_USER_SIMD_LEVEL", "NONE")

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import pandas as pd
from IPython.display import display

from author_matching import normalize_name, short_openalex_id
from project_setup import ensure_dirs, setup_project

setup = setup_project()
PROJECT = setup.project_folder

STEP_1_PREPARED = PROJECT / "step_1_data" / "prepared"
STEP_2_PREPARED = PROJECT / "step_2_data" / "prepared" / "all_papers"
STEP_3_PREPARED = PROJECT / "step_3_data" / "prepared"
STEP_3_SUMMARY = PROJECT / "step_3_artifacts" / "summary_tables"
STEP_3_CHECKS = PROJECT / "step_3_artifacts" / "check_tables"

ensure_dirs(STEP_3_PREPARED, STEP_3_SUMMARY)

PC_MEMBERS_PATH = STEP_1_PREPARED / "pc_members.parquet"
FIRST_SERVICE_PATH = STEP_1_PREPARED / "pc_first_service_evidence.parquet"
REF_AUTHORS_PATH = STEP_2_PREPARED / "all_ref_authors_exploded.parquet"
IDENTITY_TABLE_PATH = STEP_3_SUMMARY / "pc_researcher_identity_validation.csv"
MANUAL_CITATIONS_PATH = (
    STEP_3_CHECKS / "unmatched_pc_cites.csv"
)

PANEL_OUT = STEP_3_PREPARED / "panel.parquet"
PANEL_FEATURES_OUT = STEP_3_PREPARED / "panel_features.parquet"
CITATION_KEY_OUT = STEP_3_PREPARED / "panel_citation_keys.parquet"
MATCHED_HOW_OUT = STEP_3_SUMMARY / "researchers_matched_how.csv"
PANEL_SUMMARY_OUT = STEP_3_SUMMARY / "panel_summary.csv"
PANEL_IDENTITY_SUMMARY_OUT = STEP_3_SUMMARY / "panel_identity_scope_summary.csv"

print(f"Project folder: {PROJECT}")
print(f"Run mode: {setup.run_mode}")
print(f"Overwrite data: {setup.overwrite_data}")

Project folder: /Users/endersari/2026-02-citations-vs-pc-memberships
Run mode: fast
Overwrite data: False


## 2. Citation counting rules

The panel separates the row structure from the citation counting rule.
Every PC researcher stays in the panel. Citation outcomes are
filled when the identity evidence is usable:

- automatic exact name matches with one OpenAlex Author ID are used in
  the panel;
- manually accepted alternate cited author names are also used, with a
  manual flag;
- ambiguous OpenAlex ID cases and name only matches are counted by
  cited author name and flagged in the panel;
- unresolved manual review cases remain in the panel, but their
  citation outcomes are left missing for now.

## 3. Read inputs

In [2]:
pc_members = pd.read_parquet(PC_MEMBERS_PATH)
first_service = pd.read_parquet(FIRST_SERVICE_PATH)
identity = pd.read_csv(IDENTITY_TABLE_PATH)
manual_citations = pd.read_csv(MANUAL_CITATIONS_PATH)
ref_authors = pd.read_parquet(REF_AUTHORS_PATH)

print(f"pc_members:       {pc_members.shape}")
print(f"first_service:    {first_service.shape}")
print(f"identity:         {identity.shape}")
print(f"manual_citations: {manual_citations.shape}")
print(f"ref_authors:      {ref_authors.shape}")

pc_members:       (2180, 18)
first_service:    (1496, 17)
identity:         (952, 39)
manual_citations: (61, 22)
ref_authors:      (355366, 12)


## 4. Define the panel cells

I keep OOPSLA1 and OOPSLA2 separate from 2022 onward. The PC service
data still records those years as OOPSLA, so I expand the OOPSLA
2022--2025 PC rows into both OOPSLA1 and OOPSLA2 before attaching the
PC service indicator.

In [3]:
valid_cells = (
    {("ICFP", year) for year in range(2017, 2026)}
    | {("POPL", year) for year in range(2018, 2026)}
    | {("OOPSLA", year) for year in range(2017, 2022)}
    | {("OOPSLA1", year) for year in range(2022, 2026)}
    | {("OOPSLA2", year) for year in range(2022, 2026)}
    | {("PLDI", year) for year in range(2017, 2026)}
)

cells = pd.DataFrame(sorted(valid_cells), columns=["conference", "year"])

print(f"Panel cells: {len(cells)}")
display(cells.groupby("conference")["year"].agg(["min", "max", "count"]))

Panel cells: 39


,min,max,count
conference,,,
ICFP,2017,2025,9
OOPSLA,2017,2021,5
OOPSLA1,2022,2025,4
OOPSLA2,2022,2025,4
PLDI,2017,2025,9
POPL,2018,2025,8


## 5. Build the researcher list

In [4]:
people = (
    pc_members.groupby("canonical_researchr_id", as_index=False)
    .agg(
        name=("name", lambda values: values.mode().iloc[0] if not values.mode().empty else values.iloc[0]),
        n_pc_rows=("service_key", "size"),
        n_conferences=("conference", "nunique"),
        first_observed_pc_year=("year", "min"),
        last_observed_pc_year=("year", "max"),
    )
    .rename(columns={"canonical_researchr_id": "researcher_id"})
)

identity_keep = [
    "canonical_researchr_id",
    "mapped_name_for_matching",
    "openalex_name",
    "openalex_id",
    "orcid_from_openalex",
    "orcid_from_orcid_org",
    "match_method",
    "match_confidence",
    "identity_layer",
    "identity_layer_reason",
    "exploded_name_match_status",
    "candidate_openalex_ids",
    "candidate_openalex_names",
]
identity_small = identity[identity_keep].rename(
    columns={"canonical_researchr_id": "researcher_id"}
)

manual_keep = [
    "canonical_researchr_id",
    "manual_decisions",
    "accepted_ref_author_names",
    "accepted_openalex_author_ids",
    "accepted_orcids",
    "accepted_citation_rows",
    "accepted_citing_papers",
    "accepted_cited_works",
    "manual_match_status",
]
manual_small = manual_citations[manual_keep].rename(
    columns={"canonical_researchr_id": "researcher_id"}
)

researcher_list = (
    people.merge(identity_small, on="researcher_id", how="left")
    .merge(manual_small, on="researcher_id", how="left")
    .sort_values("researcher_id")
    .reset_index(drop=True)
)

researcher_list["panel_identity_status"] = "not_ready_for_main_panel"
is_automatic = researcher_list["match_method"].eq("exact_name_one_openalex_author_id")
is_manual = researcher_list["manual_match_status"].eq("matched_by_manual_review")
is_ambiguous = researcher_list["match_method"].eq("exact_name_multiple_openalex_author_ids")
is_name_only = (
    researcher_list["match_method"].eq("exact_name_no_openalex_author_id")
    | (
        researcher_list["match_method"].eq("name_not_found_in_referenced_authors")
        & researcher_list["exploded_name_match_status"].eq(
            "name_or_mapped_name_matched_with_exploded_data"
        )
    )
)

researcher_list.loc[is_automatic, "panel_identity_status"] = "automatic_openalex_id"
researcher_list.loc[is_manual, "panel_identity_status"] = "manual_accepted_cited_author_name"
researcher_list.loc[is_ambiguous, "panel_identity_status"] = "ambiguous_openalex_ids"
researcher_list.loc[is_name_only, "panel_identity_status"] = "name_only_sensitivity"
researcher_list.loc[
    researcher_list["panel_identity_status"].eq("not_ready_for_main_panel"),
    "panel_identity_status",
] = "unresolved_manual_review"

current_group_map = {
    "automatic_openalex_id": "automatic: exact name -> one OpenAlex author ID",
    "manual_accepted_cited_author_name": "manual: accepted alternate cited-author name",
    "ambiguous_openalex_ids": "automatic but ambiguous: exact name -> multiple OpenAlex IDs",
    "name_only_sensitivity": "name-only: exact/mapped cited-author name but no OpenAlex ID",
    "unresolved_manual_review": "manual review still needed",
}
recommendation_map = {
    "automatic_openalex_id": "use_in_main_panel",
    "manual_accepted_cited_author_name": "use_in_main_panel_with_manual_flag",
    "ambiguous_openalex_ids": "use_in_main_panel_with_name_based_flag",
    "name_only_sensitivity": "use_in_main_panel_with_name_based_flag",
    "unresolved_manual_review": "do_not_use_until_reviewed",
}
researcher_list["current_match_group"] = researcher_list["panel_identity_status"].map(current_group_map)
researcher_list["recommendation"] = researcher_list["panel_identity_status"].map(recommendation_map)

citation_basis_map = {
    "automatic_openalex_id": "openalex_author_id",
    "manual_accepted_cited_author_name": "manual_accepted_name",
    "ambiguous_openalex_ids": "ambiguous_name_match",
    "name_only_sensitivity": "name_only_match",
    "unresolved_manual_review": "unresolved_not_counted",
}
researcher_list["citation_match_basis"] = researcher_list["panel_identity_status"].map(citation_basis_map)
researcher_list["uses_name_based_citation_count"] = researcher_list["panel_identity_status"].isin(
    [
        "manual_accepted_cited_author_name",
        "ambiguous_openalex_ids",
        "name_only_sensitivity",
    ]
)
researcher_list["citation_identity_ready"] = ~researcher_list["panel_identity_status"].eq(
    "unresolved_manual_review"
)

matched_how_current = researcher_list[
    [
        "researcher_id",
        "name",
        "mapped_name_for_matching",
        "current_match_group",
        "recommendation",
        "citation_match_basis",
        "uses_name_based_citation_count",
        "match_method",
        "match_confidence",
        "identity_layer",
        "openalex_name",
        "openalex_id",
        "orcid_from_openalex",
        "orcid_from_orcid_org",
        "candidate_openalex_names",
        "candidate_openalex_ids",
        "manual_decisions",
        "accepted_ref_author_names",
        "accepted_openalex_author_ids",
        "accepted_orcids",
        "accepted_citation_rows",
        "accepted_citing_papers",
        "accepted_cited_works",
    ]
].rename(columns={"researcher_id": "canonical_researchr_id"})

print(f"Researchers: {len(researcher_list):,}")
print(researcher_list["panel_identity_status"].value_counts(dropna=False).to_string())
display(
    researcher_list[
        [
            "researcher_id",
            "name",
            "panel_identity_status",
            "recommendation",
            "openalex_id",
            "accepted_ref_author_names",
        ]
    ].head(10)
)

Researchers: 952
panel_identity_status
automatic_openalex_id                793
ambiguous_openalex_ids                84
manual_accepted_cited_author_name     51
name_only_sensitivity                 14
unresolved_manual_review              10


,researcher_id,name,panel_identity_status,recommendation,openalex_id,accepted_ref_author_names
0,aaronbembenek,Aaron Bembenek,ambiguous_openalex_ids,use_in_main_panel_with_name_based_flag,NaN,NaN
1,aaronstump,Aaron Stump,automatic_openalex_id,use_in_main_panel,A5072489480,NaN
2,abhinavverma1,Abhinav Verma,automatic_openalex_id,use_in_main_panel,A5101988843,NaN
3,adamchlipala,Adam Chlipala,automatic_openalex_id,use_in_main_panel,A5078100439,NaN
4,adamwelc,Adam Welc,automatic_openalex_id,use_in_main_panel,A5078668785,NaN
5,adityakanade,Aditya Kanade,ambiguous_openalex_ids,use_in_main_panel_with_name_based_flag,NaN,NaN
6,adityavthakur,Aditya V. Thakur,name_only_sensitivity,use_in_main_panel_with_name_based_flag,NaN,NaN
7,adriansampson,Adrian Sampson,automatic_openalex_id,use_in_main_panel,A5004782337,NaN
8,aggelosbiboudis,Aggelos Biboudis,automatic_openalex_id,use_in_main_panel,A5089315679,NaN
9,ahmedbouajjani,Ahmed Bouajjani,automatic_openalex_id,use_in_main_panel,A5045477471,NaN


## 6. Attach PC service indicators

In [5]:
skeleton = researcher_list.merge(cells, how="cross")
print(f"Skeleton: {skeleton.shape}")

service = pc_members[
    ["canonical_researchr_id", "conference", "year"]
].rename(columns={"canonical_researchr_id": "researcher_id"}).copy()

rolling = service["conference"].eq("OOPSLA") & service["year"].ge(2022)
service_expanded = pd.concat(
    [
        service.loc[~rolling],
        service.loc[rolling].assign(conference="OOPSLA1"),
        service.loc[rolling].assign(conference="OOPSLA2"),
    ],
    ignore_index=True,
)
service_expanded = service_expanded.merge(cells, on=["conference", "year"], how="inner")
service_expanded = service_expanded.drop_duplicates().assign(pc_member=1)

panel = skeleton.merge(
    service_expanded,
    on=["researcher_id", "conference", "year"],
    how="left",
)
panel["pc_member"] = panel["pc_member"].fillna(0).astype(int)

print(f"PC-service cells: {int(panel['pc_member'].sum()):,}")
display(panel.groupby("conference")["pc_member"].sum().rename("pc_cells"))

Skeleton: (37128, 34)
PC-service cells: 2,499


conference
ICFP       318
OOPSLA     176
OOPSLA1    348
OOPSLA2    348
PLDI       786
POPL       523
Name: pc_cells, dtype: int64

## 7. Add service history variables

In [6]:
first_per_conf = (
    panel.loc[panel["pc_member"].eq(1)]
    .groupby(["researcher_id", "conference"])["year"]
    .min()
    .rename("first_pc_year_conference")
    .reset_index()
)
first_any = (
    panel.loc[panel["pc_member"].eq(1)]
    .groupby("researcher_id")["year"]
    .min()
    .rename("first_pc_year_any")
    .reset_index()
)

panel = panel.merge(first_per_conf, on=["researcher_id", "conference"], how="left")
panel = panel.merge(first_any, on="researcher_id", how="left")
panel["ever_pc_this_conference"] = (
    panel.groupby(["researcher_id", "conference"])["pc_member"].transform("max")
).astype(int)
panel["is_first_pc_service_this_conference"] = (
    panel["pc_member"].eq(1)
    & panel["year"].eq(panel["first_pc_year_conference"])
).astype(int)

panel = panel.sort_values(["researcher_id", "conference", "year"]).reset_index(drop=True)
panel["n_prior_pc_services_this_conference"] = (
    panel.groupby(["researcher_id", "conference"])["pc_member"].cumsum()
    - panel["pc_member"]
).astype(int)

service_year_counts = (
    service_expanded.groupby(["researcher_id", "year"])
    .size()
    .rename("n_services_in_year")
    .reset_index()
)

prior_rows = []
for researcher_id, group in service_year_counts.groupby("researcher_id"):
    by_year = group.set_index("year")["n_services_in_year"].to_dict()
    years = sorted(panel.loc[panel["researcher_id"].eq(researcher_id), "year"].unique())
    running = 0
    for year in years:
        prior_rows.append(
            {
                "researcher_id": researcher_id,
                "year": year,
                "n_prior_pc_services_any_conference": running,
            }
        )
        running += int(by_year.get(year, 0))
prior_any = pd.DataFrame(prior_rows)
panel = panel.merge(prior_any, on=["researcher_id", "year"], how="left")
panel["n_prior_pc_services_any_conference"] = (
    panel["n_prior_pc_services_any_conference"].fillna(0).astype(int)
)

print("Service-history checks:")
print(panel[["pc_member", "n_prior_pc_services_this_conference", "n_prior_pc_services_any_conference"]].describe().to_string())

Service-history checks:
          pc_member  n_prior_pc_services_this_conference  n_prior_pc_services_any_conference
count  37128.000000                         37128.000000                        37128.000000
mean       0.067308                             0.157455                            0.849601
std        0.250557                             0.440909                            1.312429
min        0.000000                             0.000000                            0.000000
25%        0.000000                             0.000000                            0.000000
50%        0.000000                             0.000000                            0.000000
75%        0.000000                             0.000000                            1.000000
max        1.000000                             5.000000                           12.000000


## 8. Add first service validation fields

In [7]:
first_service_small = first_service.rename(
    columns={
        "researcher_id": "researcher_id",
        "conference": "history_conference",
    }
)[
    [
        "researcher_id",
        "history_conference",
        "first_observed_pc_year_conf",
        "source_verified_first_pc_year_conf",
        "source_verified_first_broad_service_year_conf",
        "had_prior_pc_before_observed_conf",
        "had_prior_broad_service_before_observed_conf",
        "prior_pc_evidence_count_conf",
        "prior_broad_service_evidence_count_conf",
        "is_true_first_pc_in_observed_year_conf",
        "is_true_first_broad_service_in_observed_year_conf",
    ]
]

panel["history_conference"] = panel["conference"].replace(
    {"OOPSLA1": "OOPSLA", "OOPSLA2": "OOPSLA"}
)
panel = panel.merge(
    first_service_small,
    on=["researcher_id", "history_conference"],
    how="left",
)
panel = panel.drop(columns=["history_conference"])

first_service_any_cols = [
    "first_observed_pc_year_any_target",
    "source_verified_first_pc_year_any_target",
    "source_verified_first_broad_service_year_any_target",
]
first_service_any = first_service[["researcher_id", *first_service_any_cols]].drop_duplicates()
conflicting_any_years = (
    first_service_any.groupby("researcher_id")[first_service_any_cols]
    .nunique(dropna=False)
    .gt(1)
    .any(axis=1)
)
assert not conflicting_any_years.any(), "Conflicting researcher-level first-service years"
first_service_any = first_service_any.drop_duplicates("researcher_id")
panel = panel.merge(first_service_any, on="researcher_id", how="left")

print("First-service validation rows attached:")
print(panel["source_verified_first_pc_year_conf"].notna().sum())
print("Researcher-level first-service rows attached:")
print(panel.drop_duplicates("researcher_id")["first_observed_pc_year_any_target"].notna().sum())

First-service validation rows attached:
14608
Researcher-level first-service rows attached:
952


## 9. Build citation keys

Automatic Layer 1 identities are counted by OpenAlex Author ID. Manual
accepted identities, ambiguous OpenAlex ID identities, and name only
identities are counted by normalized cited author name. The panel keeps
the match basis as an explicit flag.

In [8]:
strict_keys = researcher_list[
    researcher_list["panel_identity_status"].eq("automatic_openalex_id")
    & researcher_list["openalex_id"].notna()
][["researcher_id", "openalex_id"]].copy()
strict_keys["citation_key_type"] = "openalex_author_id"
strict_keys["citation_key_value"] = strict_keys["openalex_id"].map(short_openalex_id)
strict_keys = strict_keys[
    ["researcher_id", "citation_key_type", "citation_key_value"]
]

def split_semicolon_values(value):
    if pd.isna(value):
        return []
    return [
        item.strip()
        for item in str(value).split(";")
        if item.strip()
    ]

manual_ready = manual_citations[
    manual_citations["manual_match_status"].eq("matched_by_manual_review")
].copy()

manual_rows = []
for _, row in manual_ready.iterrows():
    for name in split_semicolon_values(row["accepted_ref_author_names"]):
        norm = normalize_name(name)
        if norm:
            manual_rows.append(
                {
                    "researcher_id": row["canonical_researchr_id"],
                    "citation_key_type": "accepted_ref_author_name_norm",
                    "citation_key_value": norm,
                }
            )

manual_keys = pd.DataFrame(manual_rows).drop_duplicates()

name_based_statuses = {
    "ambiguous_openalex_ids": "ambiguous_ref_author_name_norm",
    "name_only_sensitivity": "name_only_ref_author_name_norm",
}
name_based_rows = []
for _, row in researcher_list[
    researcher_list["panel_identity_status"].isin(name_based_statuses)
].iterrows():
    match_name = row["mapped_name_for_matching"]
    if pd.isna(match_name) or not str(match_name).strip():
        match_name = row["name"]
    norm = normalize_name(match_name)
    if norm:
        name_based_rows.append(
            {
                "researcher_id": row["researcher_id"],
                "citation_key_type": name_based_statuses[row["panel_identity_status"]],
                "citation_key_value": norm,
            }
        )

name_based_keys = pd.DataFrame(name_based_rows).drop_duplicates()
citation_keys = pd.concat([strict_keys, manual_keys, name_based_keys], ignore_index=True)
citation_keys = citation_keys.drop_duplicates()

name_key_conflicts = (
    citation_keys[
        citation_keys["citation_key_type"].str.contains("name_norm", na=False)
        | citation_keys["citation_key_type"].eq("accepted_ref_author_name_norm")
    ]
    .groupby("citation_key_value")["researcher_id"]
    .nunique()
    .reset_index(name="n_researchers")
    .query("n_researchers > 1")
)
assert name_key_conflicts.empty, "The same cited-author name key maps to multiple researchers"

print(f"Strict OpenAlex-ID keys: {len(strict_keys):,}")
print(f"Manual accepted-name keys: {len(manual_keys):,}")
print(f"Ambiguous/name-only cited-author-name keys: {len(name_based_keys):,}")
print(f"Total citation keys: {len(citation_keys):,}")
display(citation_keys.head(10))

Strict OpenAlex-ID keys: 793
Manual accepted-name keys: 89
Ambiguous/name-only cited-author-name keys: 98
Total citation keys: 980


,researcher_id,citation_key_type,citation_key_value
0,aaronstump,openalex_author_id,A5072489480
1,abhinavverma1,openalex_author_id,A5101988843
2,adamchlipala,openalex_author_id,A5078100439
3,adamwelc,openalex_author_id,A5078668785
4,adriansampson,openalex_author_id,A5004782337
5,aggelosbiboudis,openalex_author_id,A5089315679
6,ahmedbouajjani,openalex_author_id,A5045477471
7,ainalinngeorges,openalex_author_id,A5041169923
8,akashlal,openalex_author_id,A5029930688
9,akimasamorihata,openalex_author_id,A5037284935


## 10. Aggregate citation outcomes

In [9]:
refs = ref_authors.rename(
    columns={"issue": "conference", "conference_year": "year"}
).copy()
refs = refs.merge(cells, on=["conference", "year"], how="inner")
refs["ref_author_id"] = refs["ref_author_id"].map(short_openalex_id)
refs["ref_author_name_norm"] = refs["ref_author_name"].map(normalize_name)
refs["event_id"] = range(len(refs))

id_events = refs.merge(
    strict_keys,
    left_on="ref_author_id",
    right_on="citation_key_value",
    how="inner",
)
name_events = refs.merge(
    pd.concat([manual_keys, name_based_keys], ignore_index=True),
    left_on="ref_author_name_norm",
    right_on="citation_key_value",
    how="inner",
)

citation_events = pd.concat([id_events, name_events], ignore_index=True)

duplicate_event_assignments = (
    citation_events.groupby("event_id")["researcher_id"].nunique().gt(1).sum()
)
print(f"Mapped citation events: {len(citation_events):,}")
print(f"Events assigned to more than one researcher: {duplicate_event_assignments:,}")

citation_events = citation_events.drop_duplicates(
    [
        "researcher_id",
        "event_id",
    ]
)

citation_edge_cols = [
    "researcher_id",
    "conference",
    "year",
    "work_id",
    "referenced_work_id",
]
duplicate_researcher_reference_edges = (
    citation_events.groupby(citation_edge_cols, dropna=False)
    .size()
    .reset_index(name="n_rows")
    .query("n_rows > 1")
)
n_duplicate_researcher_reference_edges_removed = int(
    (duplicate_researcher_reference_edges["n_rows"] - 1).sum()
)
print(
    "Duplicate researcher/citing-paper/cited-work rows removed: "
    f"{n_duplicate_researcher_reference_edges_removed:,}"
)

def collapse_reference_edge(group):
    return pd.Series(
        {
            "is_self_citation": bool(group["is_self_citation"].astype(bool).any()),
            "is_first_author": bool((group["author_position"] == "first").any()),
            "is_first_or_last": bool(
                group["author_position"].isin(["first", "last"]).any()
            ),
        }
    )

citation_events_unique = (
    citation_events.groupby(citation_edge_cols, group_keys=False, dropna=False)
    .apply(collapse_reference_edge)
    .reset_index()
)

def aggregate_citations(group):
    return pd.Series(
        {
            "citation_count": len(group),
            "citation_count_no_self": int((~group["is_self_citation"].astype(bool)).sum()),
            "citation_count_first_author": int(group["is_first_author"].sum()),
            "citation_count_first_or_last": int(group["is_first_or_last"].sum()),
        }
    )

citation_counts = (
    citation_events_unique.groupby(["researcher_id", "conference", "year"], group_keys=False)
    .apply(aggregate_citations)
    .reset_index()
)

panel = panel.merge(
    citation_counts,
    on=["researcher_id", "conference", "year"],
    how="left",
)

outcome_cols = [
    "citation_count",
    "citation_count_no_self",
    "citation_count_first_author",
    "citation_count_first_or_last",
]
ready = panel["citation_identity_ready"]
for col in outcome_cols:
    panel.loc[ready, col] = panel.loc[ready, col].fillna(0).astype(int)

print("Citation outcome sums for identity-ready researchers:")
for col in outcome_cols:
    print(f"{col:34s} {int(panel.loc[ready, col].fillna(0).sum()):,}")

Mapped citation events: 97,443
Events assigned to more than one researcher: 0
Duplicate researcher/citing-paper/cited-work rows removed: 32
Citation outcome sums for identity-ready researchers:
citation_count                     97,411
citation_count_no_self             84,990
citation_count_first_author        28,422
citation_count_first_or_last       59,019


## 11. Save outputs

In [10]:
output_cols = [
    "researcher_id",
    "name",
    "mapped_name_for_matching",
    "openalex_id",
    "orcid_from_openalex",
    "orcid_from_orcid_org",
    "match_method",
    "match_confidence",
    "identity_layer",
    "identity_layer_reason",
    "panel_identity_status",
    "citation_identity_ready",
    "citation_match_basis",
    "uses_name_based_citation_count",
    "current_match_group",
    "recommendation",
    "conference",
    "year",
    "pc_member",
    "ever_pc_this_conference",
    "first_pc_year_conference",
    "first_pc_year_any",
    "is_first_pc_service_this_conference",
    "n_prior_pc_services_this_conference",
    "n_prior_pc_services_any_conference",
    "first_observed_pc_year_conf",
    "source_verified_first_pc_year_conf",
    "source_verified_first_broad_service_year_conf",
    "first_observed_pc_year_any_target",
    "source_verified_first_pc_year_any_target",
    "source_verified_first_broad_service_year_any_target",
    "had_prior_pc_before_observed_conf",
    "had_prior_broad_service_before_observed_conf",
    "prior_pc_evidence_count_conf",
    "prior_broad_service_evidence_count_conf",
    "is_true_first_pc_in_observed_year_conf",
    "is_true_first_broad_service_in_observed_year_conf",
    "citation_count",
    "citation_count_no_self",
    "citation_count_first_author",
    "citation_count_first_or_last",
]

panel = panel[output_cols].sort_values(
    ["researcher_id", "conference", "year"]
).reset_index(drop=True)
panel.insert(
    0,
    "panel_row_id",
    (
        panel["researcher_id"].astype(str)
        + "|"
        + panel["conference"].astype(str)
        + "|"
        + panel["year"].astype(str)
    ),
)

panel_main = panel[
    [
        "panel_row_id",
        "researcher_id",
        "name",
        "conference",
        "year",
        "pc_member",
        "citation_count",
    ]
].rename(columns={"pc_member": "pc_status"})

panel_features = panel.drop(
    columns=[
        "name",
        "pc_member",
        "citation_count",
    ]
)

panel_summary = pd.DataFrame(
    [
        {
            "n_rows": len(panel),
            "n_researchers": panel["researcher_id"].nunique(),
            "n_conference_year_cells": cells.shape[0],
            "n_pc_service_cells": int(panel["pc_member"].sum()),
            "n_identity_ready_researchers": int(
                panel.drop_duplicates("researcher_id")["citation_identity_ready"].sum()
            ),
            "n_raw_mapped_citation_rows": len(citation_events),
            "n_duplicate_researcher_reference_edges_removed": n_duplicate_researcher_reference_edges_removed,
            "n_unique_researcher_reference_edges": len(citation_events_unique),
            "n_nonzero_citation_cells": int(
                panel["citation_count"].fillna(0).gt(0).sum()
            ),
            "total_citation_count": int(panel["citation_count"].fillna(0).sum()),
        }
    ]
)

identity_scope = (
    panel.drop_duplicates("researcher_id")
    .groupby(["panel_identity_status", "citation_identity_ready"], dropna=False)
    .size()
    .rename("n_researchers")
    .reset_index()
    .sort_values("n_researchers", ascending=False)
)

def write_parquet_if_needed(frame, path, expected_rows):
    should_write = True
    reason = "new file"
    if path.exists():
        try:
            old = pd.read_parquet(path)
            old_rows = len(old)
            same = old.equals(frame.reset_index(drop=True))
        except Exception:
            old_rows = None
            same = False
        if setup.overwrite_data:
            reason = "overwrite_data=True"
        elif old_rows != expected_rows:
            reason = f"existing row count {old_rows} != expected {expected_rows}"
        elif not same:
            reason = "existing data file differs from current output"
        else:
            should_write = False
            reason = "existing data file is current"

    if should_write:
        frame.to_parquet(path, index=False)
        print(f"Wrote {path.relative_to(PROJECT)}: {frame.shape} ({reason})")
    else:
        print(f"Skipped existing data file: {path.relative_to(PROJECT)} ({reason})")

def write_csv_if_needed(frame, path):
    should_write = True
    reason = "new file"
    if path.exists():
        try:
            old = pd.read_csv(path)
            same = old.equals(frame.reset_index(drop=True))
        except Exception:
            same = False
        if setup.overwrite_artifacts:
            reason = "overwrite_artifacts=True"
        elif not same:
            reason = "existing artifact differs from current output"
        else:
            should_write = False
            reason = "existing artifact is current"

    if should_write:
        frame.to_csv(path, index=False)
        print(f"Wrote {path.relative_to(PROJECT)} ({reason})")
    else:
        print(f"Skipped existing artifact: {path.relative_to(PROJECT)} ({reason})")

write_parquet_if_needed(panel_main, PANEL_OUT, len(panel_main))
write_parquet_if_needed(panel_features, PANEL_FEATURES_OUT, len(panel_features))
write_parquet_if_needed(citation_keys, CITATION_KEY_OUT, len(citation_keys))
write_csv_if_needed(matched_how_current, MATCHED_HOW_OUT)
write_csv_if_needed(panel_summary, PANEL_SUMMARY_OUT)
write_csv_if_needed(identity_scope, PANEL_IDENTITY_SUMMARY_OUT)

display(panel_summary)
display(identity_scope)

Wrote step_3_data/prepared/panel.parquet: (37128, 7) (existing data file differs from current output)
Skipped existing data file: step_3_data/prepared/panel_features.parquet (existing data file is current)
Skipped existing data file: step_3_data/prepared/panel_citation_keys.parquet (existing data file is current)
Wrote step_3_artifacts/summary_tables/researchers_matched_how.csv (overwrite_artifacts=True)
Wrote step_3_artifacts/summary_tables/panel_summary.csv (overwrite_artifacts=True)
Wrote step_3_artifacts/summary_tables/panel_identity_scope_summary.csv (overwrite_artifacts=True)


,n_rows,n_researchers,n_conference_year_cells,n_pc_service_cells,n_identity_ready_researchers,n_raw_mapped_citation_rows,n_duplicate_researcher_reference_edges_removed,n_unique_researcher_reference_edges,n_nonzero_citation_cells,total_citation_count
0,37128,952,39,2499,942,97443,32,97411,19383,97411


,panel_identity_status,citation_identity_ready,n_researchers
1,automatic_openalex_id,True,793
0,ambiguous_openalex_ids,True,84
2,manual_accepted_cited_author_name,True,51
3,name_only_sensitivity,True,14
4,unresolved_manual_review,False,10


## 12. My Sanity checks

In [11]:
expected_rows = researcher_list["researcher_id"].nunique() * cells.shape[0]
expected_ready = int(researcher_list["citation_identity_ready"].sum())

assert len(panel) == expected_rows, (len(panel), expected_rows)
assert len(panel_main) == expected_rows, (len(panel_main), expected_rows)
assert len(panel_features) == expected_rows, (len(panel_features), expected_rows)
assert panel[["researcher_id", "conference", "year"]].duplicated().sum() == 0
assert panel_main["panel_row_id"].is_unique
assert panel_features["panel_row_id"].is_unique
assert list(panel_main.columns) == [
    "panel_row_id",
    "researcher_id",
    "name",
    "conference",
    "year",
    "pc_status",
    "citation_count",
]
assert set(panel_main["panel_row_id"]) == set(panel_features["panel_row_id"])
assert panel["researcher_id"].nunique() == researcher_list["researcher_id"].nunique()
assert cells.shape[0] == 39
assert int(panel["pc_member"].sum()) == len(service_expanded)
assert int(panel.drop_duplicates("researcher_id")["citation_identity_ready"].sum()) == expected_ready
assert duplicate_event_assignments == 0
assert (
    panel.drop_duplicates("researcher_id")["first_observed_pc_year_any_target"]
    .notna()
    .all()
), "Missing researcher-level first observed PC year"

comparable_first_years = panel[
    panel["conference"].isin(["ICFP", "POPL", "PLDI"])
    & panel["ever_pc_this_conference"].eq(1)
][["researcher_id", "conference", "first_pc_year_conference", "first_observed_pc_year_conf"]].drop_duplicates()
first_year_mismatch = comparable_first_years[
    comparable_first_years["first_observed_pc_year_conf"].gt(
        comparable_first_years["first_pc_year_conference"]
    )
]
assert first_year_mismatch.empty, (
    "Source-observed first year should not be later than the in-panel first year"
)

ready_panel = panel[panel["citation_identity_ready"]]
assert ready_panel["citation_count"].notna().all()
assert panel.loc[~panel["citation_identity_ready"], "citation_count"].isna().all()
assert citation_events_unique[citation_edge_cols].duplicated().sum() == 0

print("All panel checks passed.")
print(f"Panel rows: {len(panel):,}")
print(f"Researchers: {panel['researcher_id'].nunique():,}")
print(f"Conference-year cells: {cells.shape[0]:,}")
print(f"Identity-ready researchers: {expected_ready:,}")
print(f"Total citation_count: {int(panel['citation_count'].fillna(0).sum()):,}")

All panel checks passed.
Panel rows: 37,128
Researchers: 952
Conference-year cells: 39
Identity-ready researchers: 942
Total citation_count: 97,411
